In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from numba import njit

@njit
def advance(grid, row, col, direction, steps):
    height, width = grid.shape
    for _ in range(steps):
        color = grid[row, col]
        direction = (direction + 1 - 2 * int(color)) & 3
        grid[row, col] = color ^ 1
        if direction == 0:
            row = (row - 1) % height
        elif direction == 1:
            col = (col + 1) % width
        elif direction == 2:
            row = (row + 1) % height
        else:
            col = (col - 1) % width
    return row, col, direction

SIZE = 201
STEPS_PER_FRAME = 100
FRAMES = 240

grid = np.zeros((SIZE, SIZE), dtype=np.uint8)
state = (SIZE // 2, SIZE // 2, 0)
advance(grid, *state, 0)

fig, ax = plt.subplots(figsize=(5, 5))
image = ax.imshow(grid, cmap="binary", vmin=0, vmax=1,
                  interpolation="nearest")
ant, = ax.plot([state[1]], [state[0]], "ro", markersize=4)
title = ax.set_title("Langton's ant — 0 steps")
ax.set_axis_off()

def init():
    global state
    grid.fill(0)
    state = (SIZE // 2, SIZE // 2, 0)
    image.set_data(grid)
    ant.set_data([state[1]], [state[0]])
    title.set_text("Langton's ant — 0 steps")
    return image, ant, title

def update(frame):
    global state
    state = advance(grid, *state, STEPS_PER_FRAME)
    row, col, _ = state
    image.set_data(grid)
    ant.set_data([col], [row])
    title.set_text(f"Langton's ant — {(frame + 1) * STEPS_PER_FRAME:,} steps")
    return image, ant, title

animation = FuncAnimation(fig, update, init_func=init,
                          frames=FRAMES, interval=40,
                          blit=False, repeat=False)
plt.close(fig)
display(HTML(animation.to_jshtml(default_mode="once")))
